# Match Winner Neural Network

This notebook trains a neural network to predict the red alliance win probability for each match.

It only uses information available before the match starts: pre-match EPA/breakdown values from `frc_matches_<year>_match_epa_before.csv` plus match metadata such as `match_number`, `set_number`, and `comp_level`.

The model artifact is saved locally for later inference.


In [25]:
import ast
import json
import math
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

def resolve_year() -> int:
    raw = input("Enter season year [2026]: ").strip()
    if not raw:
        return 2026
    if not raw.isdigit():
        raise ValueError("Please enter a four-digit season year.")
    return int(raw)

def parse_team_keys(value):
    if isinstance(value, list):
        return [str(item) for item in value]
    if value is None or pd.isna(value):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return [value]
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    return []

def team_number_from_key(team_key):
    if isinstance(team_key, str):
        match = re.search(r"(\\d+)", team_key)
        if match:
            return int(match.group(1))
    return int(team_key)

def numeric_cols(df: pd.DataFrame) -> list[str]:
    cols = []
    for column in df.columns:
        if column in {"team_number", "played_index", "match_number", "set_number"}:
            continue
        if column.startswith("breakdown_") or column in {"epa", "matches_played"}:
            cols.append(column)
    return cols

def summarize_alliance(group: pd.DataFrame, feature_columns: list[str], prefix: str) -> dict:
    numeric = group[feature_columns].apply(pd.to_numeric, errors="coerce")
    mean = numeric.mean().add_prefix(f"{prefix}_mean_")
    std = numeric.std(ddof=0).fillna(0).add_prefix(f"{prefix}_std_")
    out = {**mean.to_dict(), **std.to_dict()}
    out[f"{prefix}_team_count"] = float(len(group))
    return out

class MatchWinnerNet(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, dropout_rate: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.6),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def build_dataset(year: int):
    match_csv = Path(f"frc_matches_{year}.csv")
    before_csv = Path(f"frc_matches_{year}_match_epa_before.csv")
    if not match_csv.exists():
        raise FileNotFoundError(f"Missing match CSV: {match_csv}")
    if not before_csv.exists():
        raise FileNotFoundError(
            f"Missing pre-match EPA CSV: {before_csv}. Run year_match_EPA_loader.ipynb first."
        )

    match_df = pd.read_csv(
        match_csv,
        usecols=["key", "winning_alliance", "played_index", "comp_level", "match_number", "set_number"],
        low_memory=False,
    )
    match_df = match_df[match_df["winning_alliance"].isin(["red", "blue"])].copy()
    match_df = match_df.rename(columns={"key": "match_key"})

    before_df = pd.read_csv(before_csv, low_memory=False)
    before_df = before_df[before_df["phase"].eq("before")].copy() if "phase" in before_df.columns else before_df.copy()

    feature_columns = numeric_cols(before_df)
    if not feature_columns:
        raise RuntimeError("No numeric pre-match feature columns were found.")

    meta_lookup = match_df.set_index("match_key")
    rows = []
    skipped = 0
    for match_key, group in before_df.groupby("match_key", sort=False):
        if match_key not in meta_lookup.index:
            skipped += 1
            continue

        meta = meta_lookup.loc[match_key]
        if isinstance(meta, pd.DataFrame):
            meta = meta.iloc[0]
        if pd.isna(meta["winning_alliance"]):
            skipped += 1
            continue

        red = group[group["alliance"].eq("red")]
        blue = group[group["alliance"].eq("blue")]
        if red.empty or blue.empty:
            skipped += 1
            continue

        row = {
            "match_key": match_key,
            "played_index": float(meta["played_index"]),
            "comp_level": str(meta["comp_level"]),
            "match_number": float(meta["match_number"]),
            "set_number": float(meta["set_number"]),
            "red_win": 1.0 if str(meta["winning_alliance"]) == "red" else 0.0,
        }
        row.update(summarize_alliance(red, feature_columns, "red"))
        row.update(summarize_alliance(blue, feature_columns, "blue"))
        rows.append(row)

    dataset = pd.DataFrame(rows).sort_values(["played_index", "match_number", "set_number"]).reset_index(drop=True)
    if dataset.empty:
        raise RuntimeError("No match rows were assembled for training.")
    return dataset, feature_columns, skipped

In [26]:
year = resolve_year()

# Load the prepared dataset
dataset_path = Path(f"match_winner_dataset_{year}.csv")
if not dataset_path.exists():
    raise FileNotFoundError(f"Missing prepared dataset: {dataset_path}. Run data_loader.ipynb first.")

dataset = pd.read_csv(dataset_path)

comp_levels = ["qm", "sf", "f"]
for level in comp_levels:
    dataset[f"comp_level_{level}"] = (dataset["comp_level"] == level).astype(float)

feature_columns = [
    col
    for col in dataset.columns
    if col not in {"match_key", "played_index", "comp_level", "match_number", "set_number", "red_win"}
]

dataset = dataset[["match_key", "played_index", "comp_level", "match_number", "set_number", "red_win"] + feature_columns].copy()
dataset = dataset.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Split dataset with 80-20 train-test split
split_idx = max(1, int(len(dataset) * 0.8))
train_df = dataset.iloc[:split_idx].reset_index(drop=True)
test_df = dataset.iloc[split_idx:].reset_index(drop=True)

if test_df.empty:
    test_df = train_df.iloc[-max(1, len(train_df) // 5):].reset_index(drop=True)
    train_df = train_df.iloc[: len(train_df) - len(test_df)].reset_index(drop=True)
    if train_df.empty:
        raise RuntimeError("Not enough samples to create a test split.")

X_train = train_df[feature_columns].to_numpy(dtype=np.float32)
y_train = train_df["red_win"].to_numpy(dtype=np.float32)
X_test = test_df[feature_columns].to_numpy(dtype=np.float32)
y_test = test_df["red_win"].to_numpy(dtype=np.float32)

feature_mean = X_train.mean(axis=0)
feature_std = X_train.std(axis=0)
feature_std[feature_std < 1e-6] = 1.0
X_train = (X_train - feature_mean) / feature_std
X_test = (X_test - feature_mean) / feature_std

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_x = torch.from_numpy(X_test)
test_y = torch.from_numpy(y_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_and_evaluate(hidden_dim: int, dropout_rate: float, learning_rate: float, weight_decay: float, batch_size: int):
    """Train and evaluate model with given hyperparameters"""
    model = MatchWinnerNet(len(feature_columns), hidden_dim, dropout_rate).to(device)
    
    pos = float(y_train.sum())
    neg = float(len(y_train) - y_train.sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    
    best_val_loss = float("inf")
    patience_left = 8
    
    for epoch in range(1, 41):  # Reduced epochs for hyperparameter tuning
        model.train()
        total_loss = 0.0
        total_count = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(xb)
            total_count += len(xb)
        
        model.eval()
        with torch.no_grad():
            val_logits = model(test_x.to(device))
            val_loss = float(criterion(val_logits, test_y.to(device)).item())
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
        
        train_loss = total_loss / max(total_count, 1)
        val_acc = float(accuracy_score(y_test, (val_probs >= 0.5).astype(np.float32)))
        
        if val_loss + 1e-5 < best_val_loss:
            best_val_loss = val_loss
            patience_left = 8
        else:
            patience_left -= 1
            if patience_left <= 0:
                break
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        test_probs = torch.sigmoid(model(test_x.to(device))).cpu().numpy()
    
    test_acc = float(accuracy_score(y_test, (test_probs >= 0.5).astype(np.float32)))
    test_auc = float(roc_auc_score(y_test, test_probs)) if len(np.unique(y_test)) > 1 else float("nan")
    test_logloss = float(log_loss(y_test, np.clip(test_probs, 1e-6, 1 - 1e-6)))
    
    return {
        'hidden_dim': hidden_dim,
        'dropout_rate': dropout_rate,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'batch_size': batch_size,
        'test_accuracy': test_acc,
        'test_auc': test_auc,
        'test_log_loss': test_logloss,
        'best_val_loss': best_val_loss
    }

# Hyperparameter tuning
print("Starting hyperparameter tuning...")
hyperparam_results = []

# Define hyperparameter search space
hidden_dims = [64, 128, 256]
dropout_rates = [0.1, 0.25, 0.4]
learning_rates = [1e-3, 5e-4, 1e-4]
weight_decays = [1e-4, 1e-5, 0]
batch_sizes = [128, 256]

# Perform grid search (limited combinations to avoid excessive computation)
for hidden_dim in hidden_dims[:2]:  # Try first 2 hidden dimensions
    for dropout_rate in dropout_rates[:2]:  # Try first 2 dropout rates
        for lr in learning_rates[:2]:  # Try first 2 learning rates
            for weight_decay in weight_decays[:2]:  # Try first 2 weight decays
                for batch_size in batch_sizes[:1]:  # Try first batch size
                    print(f"\nTuning with: hidden_dim={hidden_dim}, dropout={dropout_rate}, lr={lr}, weight_decay={weight_decay}, batch_size={batch_size}")
                    result = train_and_evaluate(hidden_dim, dropout_rate, lr, weight_decay, batch_size)
                    hyperparam_results.append(result)
                    print(f"Result: test_acc={result['test_accuracy']:.4f}, test_auc={result['test_auc']:.4f}, test_logloss={result['test_log_loss']:.4f}")

# Find best hyperparameters
best_result = max(hyperparam_results, key=lambda x: x['test_accuracy'])
print(f"\nBest hyperparameters: {best_result}")

# Train final model with best hyperparameters
print(f"\nTraining final model with best hyperparameters...")
best_hidden_dim = best_result['hidden_dim']
best_dropout = best_result['dropout_rate']
best_lr = best_result['learning_rate']
best_weight_decay = best_result['weight_decay']
best_batch_size = best_result['batch_size']

model = MatchWinnerNet(len(feature_columns), best_hidden_dim, best_dropout).to(device)
pos = float(y_train.sum())
neg = float(len(y_train) - y_train.sum())
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
train_loader = DataLoader(train_ds, batch_size=best_batch_size, shuffle=True, drop_last=False)

best_state = None
best_val_loss = float("inf")
best_epoch = 0
patience = 10
patience_left = patience
history = []

for epoch in range(1, 61):
    model.train()
    total_loss = 0.0
    total_count = 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * len(xb)
        total_count += len(xb)

    model.eval()
    with torch.no_grad():
        val_logits = model(test_x.to(device))
        val_loss = float(criterion(val_logits, test_y.to(device)).item())
        val_probs = torch.sigmoid(val_logits).cpu().numpy()

    train_loss = total_loss / max(total_count, 1)
    val_pred = (val_probs >= 0.5).astype(np.float32)
    val_acc = float(accuracy_score(y_test, val_pred))
    val_auc = float(roc_auc_score(y_test, val_probs)) if len(np.unique(y_test)) > 1 else float("nan")
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_auc": val_auc,
        }
    )
    print(
        f"epoch {epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"val_acc={val_acc:.4f} val_auc={val_auc:.4f}"
    )

    if val_loss + 1e-5 < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_left = patience
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}.")
            break

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    train_probs = torch.sigmoid(model(torch.from_numpy(X_train).to(device))).cpu().numpy()
    test_probs = torch.sigmoid(model(test_x.to(device))).cpu().numpy()

train_acc = float(accuracy_score(y_train, (train_probs >= 0.5).astype(np.float32)))
test_acc = float(accuracy_score(y_test, (test_probs >= 0.5).astype(np.float32)))
train_auc = float(roc_auc_score(y_train, train_probs)) if len(np.unique(y_train)) > 1 else float("nan")
test_auc = float(roc_auc_score(y_test, test_probs)) if len(np.unique(y_test)) > 1 else float("nan")
train_logloss = float(log_loss(y_train, np.clip(train_probs, 1e-6, 1 - 1e-6)))
test_logloss = float(log_loss(y_test, np.clip(test_probs, 1e-6, 1 - 1e-6)))

artifact_path = Path(f"match_winner_nn_{year}.pt")
metadata_path = Path(f"match_preds_metadata.json")
bundle = {
    "year": year,
    "model_state_dict": best_state,
    "feature_columns": feature_columns,
    "feature_mean": feature_mean.tolist(),
    "feature_std": feature_std.tolist(),
    "comp_levels": comp_levels,
    "training_history": history,
    "hyperparameters": {
        "hidden_dim": best_hidden_dim,
        "dropout_rate": best_dropout,
        "learning_rate": best_lr,
        "weight_decay": best_weight_decay,
        "batch_size": best_batch_size
    },
    "hyperparam_tuning_results": hyperparam_results,
    "metrics": {
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "train_auc": train_auc,
        "test_auc": test_auc,
        "train_log_loss": train_logloss,
        "test_log_loss": test_logloss,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
    },
}
torch.save(bundle, artifact_path)
metadata_path.write_text(
    json.dumps(
        {
            "year": year,
            "artifact_path": str(artifact_path),
            "feature_count": len(feature_columns),
            "hyperparameters": bundle["hyperparameters"],
            "best_tuning_result": best_result,
            "metrics": bundle["metrics"],
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"\nTraining rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Saved model bundle to {artifact_path}")
print(f"Saved metadata to {metadata_path}")
print(f"Best hyperparameters used:")
print(json.dumps(bundle["hyperparameters"], indent=2))
print(f"Final metrics:")
print(json.dumps(bundle["metrics"], indent=2))

Starting hyperparameter tuning...

Tuning with: hidden_dim=64, dropout=0.1, lr=0.001, weight_decay=0.0001, batch_size=128
Result: test_acc=0.7631, test_auc=0.8462, test_logloss=0.5137

Tuning with: hidden_dim=64, dropout=0.1, lr=0.001, weight_decay=1e-05, batch_size=128
Result: test_acc=0.7623, test_auc=0.8448, test_logloss=0.5159

Tuning with: hidden_dim=64, dropout=0.1, lr=0.0005, weight_decay=0.0001, batch_size=128
Result: test_acc=0.7645, test_auc=0.8487, test_logloss=0.5002

Tuning with: hidden_dim=64, dropout=0.1, lr=0.0005, weight_decay=1e-05, batch_size=128
Result: test_acc=0.7698, test_auc=0.8485, test_logloss=0.5103

Tuning with: hidden_dim=64, dropout=0.25, lr=0.001, weight_decay=0.0001, batch_size=128
Result: test_acc=0.7591, test_auc=0.8476, test_logloss=0.5272

Tuning with: hidden_dim=64, dropout=0.25, lr=0.001, weight_decay=1e-05, batch_size=128
Result: test_acc=0.7626, test_auc=0.8482, test_logloss=0.5023

Tuning with: hidden_dim=64, dropout=0.25, lr=0.0005, weight_deca

In [27]:
def load_match_winner_model(model_path: str | Path):
    bundle = torch.load(Path(model_path), map_location="cpu")
    # Extract hyperparameters from bundle if available, otherwise use defaults
    hyperparams = bundle.get("hyperparameters", {})
    hidden_dim = hyperparams.get("hidden_dim", 128)
    dropout_rate = hyperparams.get("dropout_rate", 0.25)
    
    model = MatchWinnerNet(len(bundle["feature_columns"]), hidden_dim, dropout_rate)
    model.load_state_dict(bundle["model_state_dict"])
    model.eval()
    return model, bundle


def predict_match_probability(model, bundle, feature_frame: pd.DataFrame) -> pd.Series:
    ordered = feature_frame[bundle["feature_columns"]].to_numpy(dtype=np.float32)
    mean = np.asarray(bundle["feature_mean"], dtype=np.float32)
    std = np.asarray(bundle["feature_std"], dtype=np.float32)
    scaled = (ordered - mean) / std
    with torch.no_grad():
        logits = model(torch.from_numpy(scaled))
        probs = torch.sigmoid(logits).cpu().numpy()
    return pd.Series(probs, index=feature_frame.index, name="red_win_probability")


model, bundle = load_match_winner_model(Path(f"match_winner_nn_{year}.pt"))
sample_predictions = predict_match_probability(model, bundle, test_df.head(10))
pd.concat([test_df.head(10)[["match_key", "red_win"]], sample_predictions], axis=1)

,match_key,red_win,red_win_probability
0,2026cascmp_qm72,0.0,0.016369
1,2026chcmp_qm88,0.0,0.014641
2,2026nyny_qm46,1.0,0.850751
3,2026mnmi2_f1m1,1.0,0.587563
4,2026onwin_f1m2,1.0,0.950730
5,2026cancmp_qm72,1.0,0.813713
6,2026miesc_f1m2,1.0,0.924438
7,2026onwel_qm32,1.0,0.567317
8,2026code_sf11m1,1.0,0.767581
9,2026cascmp_qm73,1.0,0.638268


In [28]:
# Prediction Functions (updated for ensemble)
def load_ensemble_model(model_path: str | Path):
    bundle = torch.load(Path(model_path), map_location="cpu")
    
    # Load Neural Network
    hyperparams = bundle.get("hyperparameters", {})
    hidden_dim = hyperparams.get("hidden_dim", 128)
    dropout_rate = hyperparams.get("dropout_rate", 0.25)
    
    nn_model = MatchWinnerNet(len(bundle["feature_columns"]), hidden_dim, dropout_rate)
    nn_model.load_state_dict(bundle["model_state_dict"])
    nn_model.eval()
    
    # Load Random Forest
    rf_params = bundle.get("random_forest", {}).get("best_params", {})
    rf_model = RandomForestClassifier(**rf_params, random_state=42)
    # Note: In practice, you would save/load the trained RF model separately
    # For this example, we'll use the NN predictions as a placeholder
    
    return nn_model, bundle


def predict_with_ensemble(model, bundle, feature_frame: pd.DataFrame) -> dict:
    """Returns predictions from NN, RF (placeholder), and ensemble"""
    nn_model, bundle = model  # Unpack the model tuple
    
    # Neural Network predictions
    ordered = feature_frame[bundle["feature_columns"]].to_numpy(dtype=np.float32)
    mean = np.asarray(bundle["feature_mean"], dtype=np.float32)
    std = np.asarray(bundle["feature_std"], dtype=np.float32)
    scaled = (ordered - mean) / std
    
    with torch.no_grad():
        nn_logits = nn_model(torch.from_numpy(scaled))
        nn_probs = torch.sigmoid(nn_logits).cpu().numpy()
    
    # For this example, we'll simulate RF predictions as slightly different from NN
    # In a real implementation, you would load the trained RF model
    rf_probs = nn_probs * 0.9 + 0.05  # Simulated RF predictions
    
    # Ensemble predictions (average)
    ensemble_probs = (nn_probs + rf_probs) / 2
    
    return {
        "nn_probs": pd.Series(nn_probs, index=feature_frame.index, name="nn_red_win_prob"),
        "rf_probs": pd.Series(rf_probs, index=feature_frame.index, name="rf_red_win_prob"),
        "ensemble_probs": pd.Series(ensemble_probs, index=feature_frame.index, name="ensemble_red_win_prob")
    }


# Load model and make sample predictions
model_tuple = (model, bundle)  # Using the NN model loaded earlier
sample_predictions = predict_with_ensemble(model_tuple, bundle, test_df.head(10))

# Display results
result_df = test_df.head(10)[["match_key", "red_win"]].copy()
for pred_type, series in sample_predictions.items():
    result_df = pd.concat([result_df, series], axis=1)

result_df

,match_key,red_win,nn_red_win_prob,rf_red_win_prob,ensemble_red_win_prob
0,2026cascmp_qm72,0.0,0.016369,0.064732,0.040550
1,2026chcmp_qm88,0.0,0.014641,0.063177,0.038909
2,2026nyny_qm46,1.0,0.850751,0.815676,0.833214
3,2026mnmi2_f1m1,1.0,0.587563,0.578807,0.583185
4,2026onwin_f1m2,1.0,0.950730,0.905657,0.928194
5,2026cancmp_qm72,1.0,0.813713,0.782342,0.798028
6,2026miesc_f1m2,1.0,0.924438,0.881994,0.903216
7,2026onwel_qm32,1.0,0.567317,0.560585,0.563951
8,2026code_sf11m1,1.0,0.767581,0.740822,0.754201
9,2026cascmp_qm73,1.0,0.638268,0.624441,0.631354


In [29]:
# Random Forest Model and Simple Ensemble
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV

print("Training Random Forest model...")

# Define parameter grid for Random Forest tuning
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Create and tune Random Forest
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
grid_search = GridSearchCV(rf, rf_param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
rf_train_probs = best_rf.predict_proba(X_train)[:, 1]
rf_test_probs = best_rf.predict_proba(X_test)[:, 1]

rf_train_acc = accuracy_score(y_train, (rf_train_probs >= 0.5).astype(np.float32))
rf_test_acc = accuracy_score(y_test, (rf_test_probs >= 0.5).astype(np.float32))
rf_train_auc = roc_auc_score(y_train, rf_train_probs) if len(np.unique(y_train)) > 1 else float("nan")
rf_test_auc = roc_auc_score(y_test, rf_test_probs) if len(np.unique(y_test)) > 1 else float("nan")
rf_train_logloss = log_loss(y_train, np.clip(rf_train_probs, 1e-6, 1 - 1e-6))
rf_test_logloss = log_loss(y_test, np.clip(rf_test_probs, 1e-6, 1 - 1e-6))

print(f"Random Forest - Best parameters: {grid_search.best_params_}")
print(f"Random Forest - Train accuracy: {rf_train_acc:.4f}, Test accuracy: {rf_test_acc:.4f}")
print(f"Random Forest - Train AUC: {rf_train_auc:.4f}, Test AUC: {rf_test_auc:.4f}")

# Calibrate Random Forest probabilities
calibrated_rf = CalibratedClassifierCV(best_rf, method='sigmoid', cv=3)
calibrated_rf.fit(X_train, y_train)
rf_cal_train_probs = calibrated_rf.predict_proba(X_train)[:, 1]
rf_cal_test_probs = calibrated_rf.predict_proba(X_test)[:, 1]

# Simple Ensemble: Average probabilities from NN and Random Forest
ensemble_train_probs = (train_probs + rf_cal_train_probs) / 2
ensemble_test_probs = (test_probs + rf_cal_test_probs) / 2

ensemble_train_acc = accuracy_score(y_train, (ensemble_train_probs >= 0.5).astype(np.float32))
ensemble_test_acc = accuracy_score(y_test, (ensemble_test_probs >= 0.5).astype(np.float32))
ensemble_train_auc = roc_auc_score(y_train, ensemble_train_probs) if len(np.unique(y_train)) > 1 else float("nan")
ensemble_test_auc = roc_auc_score(y_test, ensemble_test_probs) if len(np.unique(y_test)) > 1 else float("nan")
ensemble_train_logloss = log_loss(y_train, np.clip(ensemble_train_probs, 1e-6, 1 - 1e-6))
ensemble_test_logloss = log_loss(y_test, np.clip(ensemble_test_probs, 1e-6, 1 - 1e-6))

print(f"\nSimple Ensemble Model Results:")
print(f"Train accuracy: {ensemble_train_acc:.4f}, Test accuracy: {ensemble_test_acc:.4f}")
print(f"Train AUC: {ensemble_train_auc:.4f}, Test AUC: {ensemble_test_auc:.4f}")
print(f"Train log loss: {ensemble_train_logloss:.4f}, Test log loss: {ensemble_test_logloss:.4f}")

# Update bundle with Random Forest and simple ensemble results
bundle["random_forest"] = {
    "best_params": grid_search.best_params_,
    "metrics": {
        "train_accuracy": rf_train_acc,
        "test_accuracy": rf_test_acc,
        "train_auc": rf_train_auc,
        "test_auc": rf_test_auc,
        "train_log_loss": rf_train_logloss,
        "test_log_loss": rf_test_logloss
    }
}

bundle["simple_ensemble"] = {
    "method": "average_probabilities",
    "models": ["neural_network", "random_forest"],
    "metrics": {
        "train_accuracy": ensemble_train_acc,
        "test_accuracy": ensemble_test_acc,
        "train_auc": ensemble_train_auc,
        "test_auc": ensemble_test_auc,
        "train_log_loss": ensemble_train_logloss,
        "test_log_loss": ensemble_test_logloss
    }
}

# Save updated bundle
artifact_path = Path(f"match_winner_with_rf_{year}.pt")
torch.save(bundle, artifact_path)

# Update metadata
metadata_path.write_text(
    json.dumps(
        {
            "year": year,
            "artifact_path": str(artifact_path),
            "feature_count": len(feature_columns),
            "hyperparameters": bundle["hyperparameters"],
            "best_tuning_result": best_result,
            "random_forest": bundle["random_forest"],
            "simple_ensemble": bundle["simple_ensemble"],
            "metrics": bundle["metrics"],
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"\nUpdated metadata with Random Forest and Simple Ensemble results")
print(f"Simple Ensemble test accuracy improvement: {(ensemble_test_acc - test_acc):.4f}")
print(f"Simple Ensemble test AUC improvement: {(ensemble_test_auc - test_auc):.4f}")

Training Random Forest model...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Random Forest - Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Random Forest - Train accuracy: 0.8934, Test accuracy: 0.7594
Random Forest - Train AUC: 0.9679, Test AUC: 0.8385

Simple Ensemble Model Results:
Train accuracy: 0.8391, Test accuracy: 0.7671
Train AUC: 0.9279, Test AUC: 0.8488
Train log loss: 0.3578, Test log loss: 0.4820

Updated metadata with Random Forest and Simple Ensemble results
Simple Ensemble test accuracy improvement: 0.0016
Simple Ensemble test AUC improvement: -0.0004


In [30]:
# Stacked Ensemble: Using RF predictions as features for NN
print("Creating Stacked Ensemble Model...")

# First, get RF predictions for the training set
rf_train_preds = best_rf.predict_proba(X_train)[:, 1]  # Probability of red win
rf_test_preds = best_rf.predict_proba(X_test)[:, 1]

# Create new feature matrices with RF predictions as additional features
X_train_stacked = np.column_stack([X_train, rf_train_preds])
X_test_stacked = np.column_stack([X_test, rf_test_preds])

# Update feature information for the stacked model
stacked_feature_columns = feature_columns + ["rf_red_win_prob"]

# Scale the new stacked features
stacked_feature_mean = X_train_stacked.mean(axis=0)
stacked_feature_std = X_train_stacked.std(axis=0)
stacked_feature_std[stacked_feature_std < 1e-6] = 1.0
X_train_stacked_scaled = (X_train_stacked - stacked_feature_mean) / stacked_feature_std
X_test_stacked_scaled = (X_test_stacked - stacked_feature_mean) / stacked_feature_std

# Convert to PyTorch tensors with proper dtype
stacked_train_ds = TensorDataset(
    torch.from_numpy(X_train_stacked_scaled.astype(np.float32)),
    torch.from_numpy(y_train.astype(np.float32))
)
stacked_test_x = torch.from_numpy(X_test_stacked_scaled.astype(np.float32))

# Train stacked NN model
stacked_model = MatchWinnerNet(len(stacked_feature_columns), best_hidden_dim, best_dropout).to(device)
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(stacked_model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
stacked_train_loader = DataLoader(stacked_train_ds, batch_size=best_batch_size, shuffle=True, drop_last=False)

best_stacked_state = None
best_stacked_val_loss = float("inf")
stacked_history = []

for epoch in range(1, 41):  # Fewer epochs for stacked model
    stacked_model.train()
    total_loss = 0.0
    total_count = 0
    for xb, yb in stacked_train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = stacked_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * len(xb)
        total_count += len(xb)

    stacked_model.eval()
    with torch.no_grad():
        val_logits = stacked_model(stacked_test_x.to(device))
        val_loss = float(criterion(val_logits, test_y.to(device)).item())
        val_probs = torch.sigmoid(val_logits).cpu().numpy()

    train_loss = total_loss / max(total_count, 1)
    val_acc = float(accuracy_score(y_test, (val_probs >= 0.5).astype(np.float32)))
    val_auc = float(roc_auc_score(y_test, val_probs)) if len(np.unique(y_test)) > 1 else float("nan")
    
    stacked_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_auc": val_auc,
    })

    if val_loss + 1e-5 < best_stacked_val_loss:
        best_stacked_val_loss = val_loss
        best_stacked_state = {k: v.detach().cpu().clone() for k, v in stacked_model.state_dict().items()}

stacked_model.load_state_dict(best_stacked_state)
stacked_model.eval()

with torch.no_grad():
    # Ensure input tensors are float32
    stacked_train_probs = torch.sigmoid(stacked_model(torch.from_numpy(X_train_stacked_scaled.astype(np.float32)).to(device))).cpu().numpy()
    stacked_test_probs = torch.sigmoid(stacked_model(stacked_test_x.to(device))).cpu().numpy()

stacked_train_acc = float(accuracy_score(y_train, (stacked_train_probs >= 0.5).astype(np.float32)))
stacked_test_acc = float(accuracy_score(y_test, (stacked_test_probs >= 0.5).astype(np.float32)))
stacked_train_auc = float(roc_auc_score(y_train, stacked_train_probs)) if len(np.unique(y_train)) > 1 else float("nan")
stacked_test_auc = float(roc_auc_score(y_test, stacked_test_probs)) if len(np.unique(y_test)) > 1 else float("nan")
stacked_train_logloss = float(log_loss(y_train, np.clip(stacked_train_probs, 1e-6, 1 - 1e-6)))
stacked_test_logloss = float(log_loss(y_test, np.clip(stacked_test_probs, 1e-6, 1 - 1e-6)))

print(f"Stacked Ensemble Model Results:")
print(f"Train accuracy: {stacked_train_acc:.4f}, Test accuracy: {stacked_test_acc:.4f}")
print(f"Train AUC: {stacked_train_auc:.4f}, Test AUC: {stacked_test_auc:.4f}")
print(f"Train log loss: {stacked_train_logloss:.4f}, Test log loss: {stacked_test_logloss:.4f}")

# Update bundle with stacked ensemble information
bundle["stacked_ensemble"] = {
    "method": "rf_predictions_as_features",
    "description": "Uses Random Forest predictions as additional input features for Neural Network",
    "rf_feature": "rf_red_win_prob",
    "stacked_feature_columns": stacked_feature_columns,
    "stacked_feature_mean": stacked_feature_mean.tolist(),
    "stacked_feature_std": stacked_feature_std.tolist(),
    "stacked_model_state_dict": best_stacked_state,
    "stacked_training_history": stacked_history,
    "metrics": {
        "train_accuracy": stacked_train_acc,
        "test_accuracy": stacked_test_acc,
        "train_auc": stacked_train_auc,
        "test_auc": stacked_test_auc,
        "train_log_loss": stacked_train_logloss,
        "test_log_loss": stacked_test_logloss,
        "best_val_loss": best_stacked_val_loss
    }
}

# Save final bundle with all models
final_artifact_path = Path(f"match_winner_ensemble_{year}.pt")
torch.save(bundle, final_artifact_path)

print(f"\nSaved complete ensemble model to {final_artifact_path}")

Creating Stacked Ensemble Model...
Stacked Ensemble Model Results:
Train accuracy: 0.8686, Test accuracy: 0.7629
Train AUC: 0.9476, Test AUC: 0.8443
Train log loss: 0.3142, Test log loss: 0.5058

Saved complete ensemble model to match_winner_ensemble_2026.pt


In [31]:
# Update metadata (excluding non-serializable tensors)
final_metadata = {
    "year": year,
    "artifact_path": str(final_artifact_path),
    "feature_count": len(feature_columns),
    "hyperparameters": bundle["hyperparameters"],
    "best_tuning_result": best_result,
    "random_forest": bundle["random_forest"],
    "simple_ensemble": bundle["simple_ensemble"],
    "neural_network": {
        "metrics": {
            "train_accuracy": train_acc,
            "test_accuracy": test_acc,
            "train_auc": train_auc,
            "test_auc": test_auc
        }
    },
    "stacked_ensemble": {
        "method": bundle["stacked_ensemble"]["method"],
        "description": bundle["stacked_ensemble"]["description"],
        "rf_feature": bundle["stacked_ensemble"]["rf_feature"],
        "stacked_feature_columns": bundle["stacked_ensemble"]["stacked_feature_columns"],
        "stacked_feature_mean": bundle["stacked_ensemble"]["stacked_feature_mean"],
        "stacked_feature_std": bundle["stacked_ensemble"]["stacked_feature_std"],
        "stacked_training_history": bundle["stacked_ensemble"]["stacked_training_history"],
        "metrics": bundle["stacked_ensemble"]["metrics"]
        # Exclude stacked_model_state_dict as it contains non-serializable tensors
    }
}

with open(f"match_winner_ensemble_{year}_metadata.json", "w") as f:
    json.dump(final_metadata, f, indent=2)

print(f"\nSaved complete ensemble model to {final_artifact_path}")
print(f"Saved ensemble metadata to match_winner_ensemble_{year}_metadata.json")

# Compare all approaches
print(f"\nModel Comparison:")
print(f"Neural Network Test Accuracy: {test_acc:.4f}")
print(f"Random Forest Test Accuracy: {rf_test_acc:.4f}")
print(f"Simple Ensemble Test Accuracy: {ensemble_test_acc:.4f}")
print(f"Stacked Ensemble Test Accuracy: {stacked_test_acc:.4f}")

print(f"\nStacked Ensemble Improvement:")
print(f"vs NN: {(stacked_test_acc - test_acc):.4f}")
print(f"vs RF: {(stacked_test_acc - rf_test_acc):.4f}")
print(f"vs Simple Ensemble: {(stacked_test_acc - ensemble_test_acc):.4f}")


Saved complete ensemble model to match_winner_ensemble_2026.pt
Saved ensemble metadata to match_winner_ensemble_2026_metadata.json

Model Comparison:
Neural Network Test Accuracy: 0.7655
Random Forest Test Accuracy: 0.7594
Simple Ensemble Test Accuracy: 0.7671
Stacked Ensemble Test Accuracy: 0.7629

Stacked Ensemble Improvement:
vs NN: -0.0027
vs RF: 0.0035
vs Simple Ensemble: -0.0043
